# Test 3: does a notebook of rules, rewritten after a monthly scorecard, help?

Plan and expectations committed in `dbed894` before these answers existed: see [PLAN_2_3.md](PLAN_2_3.md).

- **Model:** `gemma-3-4b-it`. **Months:** Jan 2017 – Dec 2019, 36 questions a month (6 days × 6 markets), the same days in every run.
- **Feedback:** *none* (the twin: empty notebook), *scorecard*, *tester* (scorecard plus every rule's past record).
- **Data:** *real* markets, *random* made-up ones, *planted* ones where "reverse the 1-day move" is right 60% of the time.

This notebook only reads the saved runs. `run_feedback.py` collects them.

In [ ]:
import sys
sys.path.insert(0, "..")  # the main project's code is one folder up

import json
import re

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from evaluation import mcnemar
from llm_harness.client import SAVED

DATASETS = ["real", "random", "planted"]
KINDS = ["none", "scorecard", "tester"] 

runs = {}
for dataset in DATASETS:
    for kind in KINDS:
        path = SAVED / f"feedback_{dataset}_{kind}_state.json"
        if path.exists():
            runs[dataset, kind] = json.loads(path.read_text(encoding="utf-8"))["months"]
print("runs found:", {key: len(months) for key, months in runs.items()})

def answers(months):
    """One row per answer, with its month number (1 = the first month)."""
    rows = [dict(result, month=number + 1) for number, month in enumerate(months) for result in month["results"]]
    frame = pd.DataFrame(rows)
    frame["right"] = (frame["answer"] == "up") == frame["up"]
    return frame.set_index(["market", "day"])

runs found: {('real', 'none'): 36, ('real', 'scorecard'): 36, ('real', 'tester'): 36, ('random', 'none'): 36, ('random', 'scorecard'): 36, ('random', 'tester'): 36, ('planted', 'none'): 36, ('planted', 'scorecard'): 36, ('planted', 'tester'): 36}


## Overview

Each run over all its months. **Late minus early** compares accuracy in months 25–36 with months 1–12.

In [2]:
rows = {}
for (dataset, kind), months in runs.items():
    a = answers(months)
    early, late = a[a["month"] <= 12], a[a["month"] > len(months) - 12]
    rows[dataset, kind] = {
        "answers": len(a),
        "accuracy %": 100 * a["right"].mean(),
        "saying up every time %": 100 * a["up"].mean(),
        "AUC": roc_auc_score(a["up"], a["p_up"]) if a["up"].nunique() == 2 else np.nan,
        "late minus early, points": 100 * (late["right"].mean() - early["right"].mean()),
        "used 'reverse the 1-day move' late %": 100 * (late["rule"] == "reverse the 1-day move").mean(),
    }
pd.DataFrame(rows).T.round(3)

answers  accuracy %  saying up every time %    AUC  \
real    none        1296.0      50.926                  51.003  0.511   
        scorecard   1296.0      51.312                  51.003  0.529   
        tester      1296.0      51.157                  51.003  0.482   
random  none        1296.0      50.694                  50.772  0.489   
        scorecard   1296.0      51.003                  50.772  0.500   
        tester      1296.0      49.614                  50.772  0.503   
planted none        1296.0      49.228                  49.846  0.466   
        scorecard   1296.0      49.769                  49.846  0.496   
        tester      1296.0      50.463                  49.846  0.515   

                   late minus early, points  \
real    none                         -5.324   
        scorecard                    -5.324   
        tester                       -6.481   
random  none                          2.546   
        scorecard                     2.315   
        tester                       -0.231   
planted none                          6.250   
        scorecard                     8.102   
        tester                        5.324   

                   used 'reverse the 1-day move' late %  
real    none                                      0.000  
        scorecard                                 0.463  
        tester                                    0.000  
random  none                                      0.000  
        scorecard                                 0.231  
        tester                                   89.583  
planted none                                      0.000  
        scorecard                                 1.389  
        tester                                  100.000

## Main result: better than its twin?

Each feedback run against the *none* run on the same days. McNemar's test counts days where only one of the two was right.

In [3]:
rows = {}
for dataset in DATASETS:
    if (dataset, "none") not in runs:
        continue
    twin = answers(runs[dataset, "none"])
    for kind in ["scorecard", "tester"]:
        if (dataset, kind) not in runs:
            continue
        both = answers(runs[dataset, kind]).join(twin[["right"]].add_suffix(" twin"), how="inner")
        only_run, only_twin, p = mcnemar(pd.Series(True, index=both.index), both["right"], both["right twin"])
        rows[dataset, kind] = {
            "days": len(both),
            "accuracy %": 100 * both["right"].mean(),
            "twin accuracy %": 100 * both["right twin"].mean(),
            "difference, points": 100 * (both["right"].mean() - both["right twin"].mean()),
            "p": p,
        }
pd.DataFrame(rows).T.round(3)

days  accuracy %  twin accuracy %  difference, points  \
real    scorecard  1296.0      51.312           50.926               0.386   
        tester     1296.0      51.157           50.926               0.231   
random  scorecard  1296.0      51.003           50.694               0.309   
        tester     1296.0      49.614           50.694              -1.080   
planted scorecard  1296.0      49.769           49.228               0.540   
        tester     1296.0      50.463           49.228               1.235   

                       p  
real    scorecard  0.603  
        tester     0.784  
random  scorecard  0.704  
        tester     0.685  
planted scorecard  0.401  
        tester     0.591

## Month by month

Accuracy in each month (%), one column per run. 36 answers a month, so single months are noisy.

In [4]:
monthly = pd.DataFrame({f"{dataset} {kind}": answers(months).groupby("month")["right"].mean() * 100
                        for (dataset, kind), months in runs.items()})
monthly.round(0)

,real none,real scorecard,real tester,random none,random scorecard,random tester,planted none,planted scorecard,planted tester
month,,,,,,,,,
1,61.0,61.0,61.0,47.0,47.0,47.0,42.0,42.0,42.0
2,56.0,58.0,56.0,33.0,31.0,58.0,47.0,47.0,50.0
3,56.0,58.0,58.0,61.0,56.0,31.0,50.0,47.0,33.0
4,64.0,64.0,64.0,58.0,61.0,53.0,61.0,61.0,56.0
5,69.0,69.0,69.0,50.0,50.0,42.0,42.0,42.0,58.0
6,31.0,33.0,39.0,44.0,44.0,56.0,42.0,44.0,33.0
7,56.0,58.0,61.0,44.0,50.0,56.0,61.0,64.0,56.0
8,61.0,61.0,61.0,58.0,61.0,39.0,53.0,39.0,33.0
9,56.0,53.0,56.0,47.0,47.0,56.0,44.0,47.0,47.0


## The notebooks

**Made-up numbers:** percentages in a notebook that are within 1 point of no number in any scorecard the model had seen so far. **Churn:** share of months where the notebook changed.

In [5]:
def made_up(months):
    """Percentages written into the notebooks, and how many match nothing in the scorecards seen so far."""
    seen, claimed, unsupported = [], 0, 0
    for month in months:
        seen += [float(n) for n in re.findall(r"\d+(?:\.\d+)?", month["scorecard"])]
        for number in re.findall(r"(\d+(?:\.\d+)?)\s*%", month["notebook after"]):
            claimed += 1
            unsupported += not any(abs(float(number) - s) <= 1 for s in seen)
    return claimed, unsupported

rows = {}
for (dataset, kind), months in runs.items():
    if kind == "none":
        continue
    claimed, unsupported = made_up(months)
    notebooks = [m["notebook after"] for m in months]
    rows[dataset, kind] = {
        "percentages written": claimed,
        "made up %": 100 * unsupported / claimed if claimed else np.nan,
        "months the notebook changed %": 100 * np.mean([a != b for a, b in zip([months[0]["notebook used"]] + notebooks, notebooks)]),
    }
pd.DataFrame(rows).T.round(1)

percentages written  made up %  \
real    scorecard                365.0        0.0   
        tester                   296.0        0.0   
random  scorecard                394.0        6.9   
        tester                   378.0        0.0   
planted scorecard                342.0        7.6   
        tester                   329.0       27.7   

                   months the notebook changed %  
real    scorecard                          100.0  
        tester                             100.0  
random  scorecard                           97.2  
        tester                             100.0  
planted scorecard                          100.0  
        tester                              44.4

The last notebook of each run, as the model wrote it.

In [6]:
for (dataset, kind), months in runs.items():
    if kind != "none":
        print(f"--- {dataset}, {kind}, after {months[-1]['month']}:\n{months[-1]['notebook after']}\n")

--- real, scorecard, after 2019-12:
**Forecasting Rules – Updated** 1. **Core Strategy – 1-Day Move:** Remains central, 42% accuracy – foundational baseline. 2. **Low Confidence (<80%):** Predict “up” with 71% accuracy (improved). 3. **High Confidence (80%+):** Predict “down” with 33% accuracy – significantly reduced; prioritize alternative strategies. 4. **Strategic Adjustment:** February 2018 risk aversion remains a factor; maintain caution. 5. **Follow 1-Day Move:** Used 36 times, right 18 – refine further, but with caution, particularly below 80% confidence. 6. **Confidence Threshold:** Below 80% – predict “up” (50% accuracy, 27 answers) – highly effective. 80%+ – predict “down” (33% accuracy, 9 answers) – needs substantial adjustment. 7. **Reverse 1-Day Move:** Used 1 time, right 1 – explore cautiously. 8. **None of these:** Used 1 time, right 1 – viable alternative. Revised Targets: 1-Day Move – 42%, Low Confidence – 71%, High Confidence – 33%. Prioritizing low-confidence predict

## Added after looking: does it do what its rule says?

Not in the plan. In the planted tester run it named "reverse the 1-day move" for every late answer but barely beat its twin. So, for months 25–36: how often its answer matches what its named rule gives, and how it would have scored if it had simply obeyed that rule. Answers with "none of these" are left out.

In [7]:
from llm_harness import rules
from llm_harness.questions import days_before
from llm_harness.run_feedback import load

rows = {}
for dataset in DATASETS:
    markets = load(dataset)
    for kind in KINDS:
        late = answers(runs[dataset, kind]).reset_index()
        late = late[late["month"] > 24]
        late["rule's answer"] = [rules.implied_answer(days_before(markets[m], pd.Timestamp(d)), rule)
                                 for m, d, rule in zip(late["market"], late["day"], late["rule"])]
        named = late.dropna(subset=["rule's answer"])
        truth = named["up"].map({True: "up", False: "down"})
        rows[dataset, kind] = {
            "answers": len(named),
            "said up %": 100 * (named["answer"] == "up").mean(),
            "answer agrees with its own rule %": 100 * (named["answer"] == named["rule's answer"]).mean(),
            "accuracy %": 100 * named["right"].mean(),
            "accuracy if it had obeyed its rule %": 100 * (named["rule's answer"] == truth).mean(),
        }
pd.DataFrame(rows).T.round(1)

answers  said up %  answer agrees with its own rule %  \
real    none         432.0       95.4                               55.3   
        scorecard    431.0       98.8                               50.1   
        tester       432.0      100.0                               50.7   
random  none         432.0       97.0                               53.0   
        scorecard    432.0       99.5                               50.7   
        tester       432.0       21.5                               49.5   
planted none         432.0       95.6                               52.3   
        scorecard    428.0       98.8                               47.2   
        tester       432.0       35.2                               48.4   

                   accuracy %  accuracy if it had obeyed its rule %  
real    none             50.2                                  50.0  
        scorecard        50.1                                  50.3  
        tester           49.8                                  50.0  
random  none             53.5                                  50.0  
        scorecard        53.7                                  49.8  
        tester           47.9                                  50.2  
planted none             53.5                                  36.3  
        scorecard        54.4                                  37.6  
        tester           51.4                                  63.7

## Added after looking: a stricter made-up-numbers check

The planned check accepts a percentage if it is within 1 point of *any* number in *any* scorecard seen so far, counts included. By month 36 that covers most whole numbers from 0 to 100, so it can hardly fail. Two stricter versions: against percentages only (any scorecard seen so far), and against this month's scorecard only. The last is too strict for numbers the notebook kept from earlier months.

In [8]:
PERCENT = r"(\d+(?:\.\d+)?)\s*%"

def share_made_up(months, pool):
    """% of the notebooks' percentages within 1 point of nothing in pool(months up to now)."""
    claimed = unsupported = 0
    for number, month in enumerate(months):
        allowed = pool(months[: number + 1])
        for value in re.findall(PERCENT, month["notebook after"]):
            claimed += 1
            unsupported += not any(abs(float(value) - a) <= 1 for a in allowed)
    return 100 * unsupported / claimed

any_number = lambda ms: [float(n) for m in ms for n in re.findall(r"\d+(?:\.\d+)?", m["scorecard"])]
any_percent = lambda ms: [float(n) for m in ms for n in re.findall(PERCENT, m["scorecard"])]
this_month = lambda ms: [float(n) for n in re.findall(PERCENT, ms[-1]["scorecard"])]

rows = {}
for (dataset, kind), months in runs.items():
    if kind != "none":
        rows[dataset, kind] = {
            "whole numbers 0-100 the planned check accepts by month 36": sum(any(abs(x - a) <= 1 for a in any_number(months)) for x in range(101)),
            "made up %, planned check": share_made_up(months, any_number),
            "vs percentages seen so far": share_made_up(months, any_percent),
            "vs this month's percentages": share_made_up(months, this_month),
        }
pd.DataFrame(rows).T.round(1)

whole numbers 0-100 the planned check accepts by month 36  \
real    scorecard                                               83.0           
        tester                                                  80.0           
random  scorecard                                               75.0           
        tester                                                  76.0           
planted scorecard                                               77.0           
        tester                                                  72.0           

                   made up %, planned check  vs percentages seen so far  \
real    scorecard                       0.0                        17.3   
        tester                          0.0                        13.2   
random  scorecard                       6.9                        34.3   
        tester                          0.0                        38.6   
planted scorecard                       7.6                        49.4   
        tester                         27.7                        51.4   

                   vs this month's percentages  
real    scorecard                         67.7  
        tester                            49.0  
random  scorecard                         75.1  
        tester                            68.5  
planted scorecard                         78.1  
        tester                            63.8

## Conclusions

**Feedback did not make it better.** Against its twin on the same days, every feedback run is within 1.3 points (p 0.40 to 0.78). Real: +0.4 and +0.2. Random: +0.3 and −1.1. Planted: +0.5 and +1.2. AUC stays between 0.47 and 0.53 everywhere.

| Expectation from the plan | What happened |
|---|---|
| real, either feedback: no difference | As expected |
| random, either feedback: no difference | As expected |
| planted, scorecard: little or none | As expected. It used "reverse the 1-day move" in only 1.4% of late answers |
| planted, tester: switches within 6 months, beats its twin by 5 points or more | **Half right.** It switched after the first month and named the rule in 100% of late answers. But it beat its twin by only 1.2 points (p 0.59) |
| made-up numbers: common | The planned check found 0–28%, but it is too loose (below). Stricter checks: common |
| stubborn: one rule unless the tester shows a better one | Yes in real data. With the tester on made-up data it switched after one month, even on noise (below) |

**Late minus early is not learning.** The twins, which never see feedback, move just as much (real −5.3, planted +6.3). The change comes from which days fell in those months.

**It names the right rule but doesn't use it (added after looking).** In the planted tester run, obeying "reverse the 1-day move" would have scored 63.7% in months 25–36. Its answers matched that rule only 48.4% of the time, a coin flip, so it scored 51.4%. In every run its answer matches its own rule 47–55% of the time. Test 2 found the same (64.7%, mostly from saying "up").

**It switches on noise.** On random data, the first tester showed "reverse the 1-day move" at 50.1% against 49.9% for its default "follow the 1-day move". That gap is noise, and it wasn't even the top rule ("reverse the 20-day move" showed 50.4%). The model still switched after one month and used it in 90% of late answers. On real data it kept "follow the 1-day move" throughout.

**Made-up numbers (stricter check added after looking).** By month 36 the planned check accepts 72–83 of the 101 whole numbers from 0 to 100, so it can hardly fail. Against percentages it was actually shown in any scorecard, 13–51% of its percentages match nothing. Real examples from its last notebooks: "1-Day Move: Used 36 times, right 18. Guaranteed 100% accuracy", and "Reverse the 1-day move: Prioritized (72% accuracy)" when the tester said about 60%.

**Churn.** It rewrote the notebook nearly every month (97–100%), except in the planted tester run (44%), where it settled on the rule early.

**In short:** a 4-billion-parameter model with a notebook and a monthly scorecard does not learn to forecast, even when the data contain a simple rule that works 60% of the time and it is told which rule that is. It copies the rule's name into its notebook, but its answers don't follow it.